<a href="https://colab.research.google.com/github/juliaR0/Baseline/blob/main/notebooks/S_Chunk3_Pilot_Lesion_112326.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chunk 3a summary — Pilot lesion (Nov 23, 2026)

*Summarized with Claude AI

**Goal:** Damage the grammar path of a trained twin and see whether its errors look like agrammatism, not just general damage.

**Method (prototype):**
- Twin 1, healthy weights at epoch 20 (`w20.lz4`)
- `lesion()`: copy the layers, zero a random fraction of the weights at one site (seed = repeatable), save
- Sites: **grammar_A** = hidden → pred_compress; **control_meaning** = hidden → pred_role
- Test without learning: `--epochs 1 --sw <folder> --swe 20 --input simulations/results --norandomization --hidden 74 --compress 56 --fw 19`, **epoch 0 rows only**
- Sanity check: 0% copy = 85.8% meaning (healthy 86.0%) → pipeline works

**Results (meaning % by structure):**
| | Intrans. | Trans. | Ditrans. | Grammatical % |
|---|---|---|---|---|
| Healthy | 83 | 90 | 77 | 97 |
| Grammar 40–50% (typical of 4/5 seeds) | ~80 | ~85 | **21–48** | 85–97 |
| Meaning 40–50% | **26–29** | 47–49 | 50 | 91 |
| Grammar 60–95% | collapse | collapse | collapse | 14–63 |

**Findings:**
- Grammar damage hits **3-argument (ditransitive) sentences first**, e.g. dropped recipients (*he throws a toy to the waitress* → *he throws a toy*). This matches the argument-structure hierarchy in agrammatism.
- Meaning damage gives the **opposite shape** (simple sentences hit hardest, grammar intact) → start of a **double dissociation**.
- Heavy grammar damage (≥60%) = general collapse, not agrammatism.
- Article/tense-only errors (flex-False) did **not** increase.

**Limits:** 1 twin, few seeds, ~50 ditransitive sentences per run, no statistics, no language split yet.

**Next:** more twins/seeds, tense scorer, English vs Spanish, grammar_B site.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/dual_path

Mounted at /content/drive
/content/drive/MyDrive/dual_path


In [2]:
!sed 's/==.*//' requirements.txt > requirements_unpinned.txt
!pip install -q -r requirements_unpinned.txt

In [3]:
import sys, os, copy, pickle, lz4.frame, torch
sys.path.insert(0, 'src')

with lz4.frame.open('simulations/results/1/weights/w20.lz4', 'rb') as f:
    w = pickle.load(f)
print(len(w), "layers loaded")

13 layers loaded


In [4]:
LESIONS = {
    'grammar_A': ('pred_compress', slice(0, 74)),     # hidden -> pred_compress
    'grammar_B': ('output',        slice(55, 111)),   # pred_compress -> output
    'control_meaning': ('pred_role', slice(0, 74)),   # hidden -> pred_role
}

def lesion(layers, site, fraction, seed=0):
    damaged = copy.deepcopy(layers)
    layer_name, rows = LESIONS[site]
    W = damaged[layer_name].in_weights
    g = torch.Generator().manual_seed(seed)
    mask = torch.rand(W[rows].shape, generator=g) < fraction
    W[rows] = W[rows].masked_fill(mask, 0.0)
    return damaged

def save_layers(layers, folder, sim, epoch=20):
    path = f'{folder}/{sim}/weights'
    os.makedirs(path, exist_ok=True)
    with lz4.frame.open(f'{path}/w{epoch}.lz4', 'wb') as f:
        pickle.dump(layers, f, protocol=-1)

In [5]:
d = lesion(w, 'grammar_A', 0.4)
before = (w['pred_compress'].in_weights == 0).float().mean().item()
after  = (d['pred_compress'].in_weights == 0).float().mean().item()
print(f"zeros before: {before:.1%}, after: {after:.1%}")   # expect ~0% -> ~40%

save_layers(lesion(w, 'grammar_A', 0.0), 'simulations/lesion_grammarA_00', sim=1)   # 0% = sanity check
save_layers(d,                           'simulations/lesion_grammarA_40', sim=1)   # 40% damage
!ls simulations/lesion_grammarA_00/1/weights simulations/lesion_grammarA_40/1/weights

zeros before: 0.0%, after: 40.3%
simulations/lesion_grammarA_00/1/weights:
w20.lz4

simulations/lesion_grammarA_40/1/weights:
w20.lz4


In [7]:
import os, shutil
src_twin  = 'simulations/results/1'        # twin 1's sentences
src_input = 'simulations/results/input'    # lexicon, structures, etc.
dst = 'simulations/input_twin1'
os.makedirs(dst, exist_ok=True)

for folder in [src_input, src_twin]:
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        if os.path.isfile(p) and not f.endswith('.csv') or f in ('lexicon.csv', 'structures.csv'):
            shutil.copyfile(p, os.path.join(dst, f))

!ls simulations/input_twin1

concepts.in	    identifiability.in	lexicon.in  structures.csv  test.in
event_semantics.in  lexicon.csv		roles.in    target_lang.in  training.in


In [8]:
#!python src/start_dual_path.py --evaluate --norandomization --sim 1 --hidden 74 --compress 56 --fw 19 --cs --lang es en --sw simulations/lesion_grammarA_00 --swe 20 --input simulations/input_twin1 --resdir eval_grammarA_00_v2

INFO:root:Predefined input folder (simulations/input_twin1), will use that instead of generating a new set
Traceback (most recent call last):
  File "/content/drive/MyDrive/dual_path/src/start_dual_path.py", line 569, in <module>
    copy_files(
    ~~~~~~~~~~^
        src=os.path.join(existing_input_path, str(sim)),
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        dest=f"{results_dir}/{sim}",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        ends_with=".in",
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "/content/drive/MyDrive/dual_path/src/modules/__init__.py", line 217, in copy_files
    for filename in os.listdir(src):
                    ~~~~~~~~~~^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'simulations/input_twin1/1'


In [9]:
os.makedirs('simulations/input_twin1/1', exist_ok=True)
for f in os.listdir('simulations/results/1'):
    if f.endswith('.in'):
        shutil.copyfile(f'simulations/results/1/{f}', f'simulations/input_twin1/1/{f}')
!ls simulations/input_twin1/1

test.in  training.in


In [11]:
#!python src/start_dual_path.py --evaluate --norandomization --sim 1 --hidden 74 --compress 56 --fw 19 --cs --lang es en --sw simulations/lesion_grammarA_00 --swe 20 --input simulations/results --resdir eval_grammarA_00_v4

INFO:root:Predefined input folder (simulations/results/input), will use that instead of generating a new set
Traceback (most recent call last):
  File "/content/drive/MyDrive/dual_path/src/start_dual_path.py", line 715, in <module>
    dualp = DualPath(
        hidden_size=args.hidden,
    ...<28 lines>...
        priming_experiment=args.priming_experiment,
    )
  File "/content/drive/MyDrive/dual_path/src/modules/dual_path.py", line 130, in __init__
    self.initialize_srn()
    ~~~~~~~~~~~~~~~~~~~^^
  File "/content/drive/MyDrive/dual_path/src/modules/dual_path.py", line 279, in initialize_srn
    self.srn.load_weights(
    ~~~~~~~~~~~~~~~~~~~~~^
        set_weights_epoch=self.set_weights_epoch,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        set_weights_folder=self.set_weights_folder,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        simulation_num=self.simulation_num,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/content/drive/MyDrive/dual_pa

In [12]:
for folder in ['simulations/lesion_grammarA_00', 'simulations/lesion_grammarA_40']:
    os.makedirs(f'{folder}/weights', exist_ok=True)
    shutil.copyfile(f'{folder}/1/weights/w20.lz4', f'{folder}/weights/w20.lz4')
!ls simulations/lesion_grammarA_00 simulations/lesion_grammarA_00/weights

simulations/lesion_grammarA_00:
1  weights

simulations/lesion_grammarA_00/weights:
w20.lz4


In [13]:
#!python src/start_dual_path.py --evaluate --norandomization --sim 1 --hidden 74 --compress 56 --fw 19 --cs --lang es en --sw simulations/lesion_grammarA_00 --swe 20 --input simulations/results --resdir eval_grammarA_00_v5

INFO:root:Predefined input folder (simulations/results/input), will use that instead of generating a new set
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py", line 490, in _process_worker
    r = call_item()
  File "/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/joblib/parallel.py", line 607, in __call__
    return [func(*args, **kwargs) for func, args, kwargs in self.items]
            ~~~~^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/dual_path/src/modules/dual_path.py", line 498, in start_network
    self.evaluate_network(set_names=set_names)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/dual_path/src/modules/dual_pat

In [14]:
!python src/start_dual_path.py --norandomization --sim 1 --epochs 20 --hidden 74 --compress 56 --fw 19 --cs --lang es en --sw simulations/lesion_grammarA_00 --swe 20 --input simulations/results --resdir eval_grammarA_00_v6

INFO:root:Predefined input folder (simulations/results/input), will use that instead of generating a new set
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/joblib/parallel.py", line 1704, in _get_outputs
    yield from self._retrieve()
  File "/usr/local/lib/python3.13/dist-packages/joblib/parallel.py", line 1822, in _retrieve
    time.sleep(0.01)
    ~~~~~~~~~~^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/dual_path/src/start_dual_path.py", line 748, in <module>
    Parallel(n_jobs=-1)(
    ~~~~~~~~~~~~~~~~~~~^
        delayed(dualp.start_network)(
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
        for sim in simulation_range
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/joblib/parallel.py", line 2098, in __call__
    return output if self.return_generator else list(output)

In [16]:
!python src/start_dual_path.py --norandomization --sim 1 --epochs 1 --hidden 74 --compress 56 --fw 19 --cs --lang es en --sw simulations/lesion_grammarA_00 --swe 20 --input simulations/results --resdir eval_grammarA_00_v7

INFO:root:Predefined input folder (simulations/results/input), will use that instead of generating a new set


In [17]:
import pandas as pd
t = pd.read_csv('simulations/eval_grammarA_00_v7/1/test.csv')
print(t.epoch.unique())
t0 = t[t.epoch == 0]
print(len(t0), "sentences at epoch 0")
print((t0.meaning.astype(str).str.strip() == 'True').mean())   # hoping ~0.86

[0 1]
500 sentences at epoch 0
0.858


In [18]:
!python src/start_dual_path.py --norandomization --sim 1 --epochs 1 --hidden 74 --compress 56 --fw 19 --cs --lang es en --sw simulations/lesion_grammarA_40 --swe 20 --input simulations/results --resdir eval_grammarA_40_v1

INFO:root:Predefined input folder (simulations/results/input), will use that instead of generating a new set


In [19]:
def load0(path):
    d = pd.read_csv(path)
    d = d[d.epoch == 0].copy()
    d['m'] = d['meaning'].astype(str).str.strip()
    for col in ['is_grammatical', 'is_code_switched']:
        d[col] = d[col].astype(str).str.strip() == 'True'
    d['lang'] = d.target_sentence.str.split().str[0].isin(['el','la','un','una','él','ella']).map({True:'es', False:'en'})
    return d

rows = []
for name, path in [('healthy (0%)', 'simulations/eval_grammarA_00_v7/1/test.csv'),
                   ('grammar_A 40%', 'simulations/eval_grammarA_40_v1/1/test.csv')]:
    d = load0(path)
    for lang, g in [('all', d)] + list(d.groupby('lang')):
        rows.append({'model': name, 'lang': lang,
                     'meaning_%': round(100 * (g.m == 'True').mean(), 1),
                     'flex_False': (g.m == 'flex-False').sum(),
                     'grammatical_%': round(100 * g.is_grammatical.mean(), 1),
                     'switch_%': round(100 * g.is_code_switched.mean(), 1)})
print(pd.DataFrame(rows))

dmg = load0('simulations/eval_grammarA_40_v1/1/test.csv')
pd.set_option('display.max_colwidth', 80)
dmg[dmg.m != 'True'][['m', 'target_sentence', 'produced_sentence']].head(20)

           model lang  meaning_%  flex_False  grammatical_%  switch_%
0   healthy (0%)  all       85.8           7           97.0      23.8
1   healthy (0%)   en       94.6           7           99.7       2.2
2   healthy (0%)   es       70.8           0           92.4      60.5
3  grammar_A 40%  all       83.8           5           96.6      26.6
4  grammar_A 40%   en       88.6           5           99.0       6.0
5  grammar_A 40%   es       75.7           0           92.4      61.6


,m,target_sentence,produced_sentence
18,flex-False,a gentleman shows a book to the waitress .,a gentleman shows the book to the waitress .
22,False,a sad waitress throws the book to a aunt .,a sad throws throws the book to a aunt .
32,False,the short boy pushes the key .,the short boy has the key .
43,False,he throws a toy to the waitress .,he throws a toy .
66,False,the aunt ran .,the aunt ate .
68,False,un tío pequeño tira a_ un tío una pelota .,a short uncle throws a ball .
71,False,ella muestra a_ una niña el juguete .,ella muestra shows toy .
79,False,él ha pateado un bolígrafo .,he has kicked a toy .
92,False,she shows a sister a book .,she shows shows book .
98,False,un padre tira a_ el abuelo un juguete .,a father throws a grandfather the grandfather .


In [20]:
def run_lesion(site, fraction, seed=0, sim=1):
    tag = f"{site}_{int(fraction*100):02d}_s{seed}"
    folder = f"simulations/lesion_{tag}"
    dmg = lesion(w, site, fraction, seed)
    save_layers(dmg, folder, sim)
    os.makedirs(f"{folder}/weights", exist_ok=True)
    shutil.copyfile(f"{folder}/{sim}/weights/w20.lz4", f"{folder}/weights/w20.lz4")
    !python src/start_dual_path.py --norandomization --sim 1 --epochs 1 --hidden 74 --compress 56 --fw 19 --cs --lang es en --sw {folder} --swe 20 --input simulations/results --resdir eval_{tag} > /dev/null 2>&1
    return f"simulations/eval_{tag}/1/test.csv"

paths = {0.0: 'simulations/eval_grammarA_00_v7/1/test.csv',
         0.4: 'simulations/eval_grammarA_40_v1/1/test.csv'}
for frac in [0.6, 0.8, 0.95]:
    paths[frac] = run_lesion('grammar_A', frac)
    print("done", frac)

done 0.6
done 0.8
done 0.95


In [21]:
def structure(msg):
    msg = str(msg)
    if 'RECIPIENT=' in msg: return '3 ditransitive'
    if 'PATIENT=' in msg:   return '2 transitive'
    if 'ATTR=' in msg:      return 'copular'
    return '1 intransitive'

rows = []
for frac, path in paths.items():
    d = load0(path)
    d['struct'] = d.message.apply(structure)
    for s, g in d.groupby('struct'):
        rows.append({'damage': frac, 'structure': s, 'meaning_%': round(100 * (g.m == 'True').mean(), 1)})
    rows.append({'damage': frac, 'structure': 'ALL grammatical_%', 'meaning_%': round(100 * d.is_grammatical.mean(), 1)})
    rows.append({'damage': frac, 'structure': 'ALL flex_False', 'meaning_%': (d.m == 'flex-False').sum()})
pd.DataFrame(rows).pivot(index='structure', columns='damage', values='meaning_%')

damage,0.00,0.40,0.60,0.80,0.95
structure,,,,,
1 intransitive,83.1,89.6,29.5,35.0,1.6
2 transitive,90.0,90.0,11.4,37.4,0.0
3 ditransitive,76.9,25.0,3.8,13.5,0.0
ALL flex_False,7.0,5.0,1.0,7.0,0.0
ALL grammatical_%,97.0,96.6,27.4,62.8,13.8
copular,87.0,97.8,50.0,76.1,23.9


In [22]:
more = {}
for seed in [1, 2]:
    more[('grammar_A', 0.4, seed)] = run_lesion('grammar_A', 0.4, seed)
    more[('grammar_A', 0.5, seed)] = run_lesion('grammar_A', 0.5, seed)
for frac in [0.4, 0.5]:
    more[('control_meaning', frac, 0)] = run_lesion('control_meaning', frac, 0)
print("done")

rows = []
for (site, frac, seed), path in more.items():
    d = load0(path)
    d['struct'] = d.message.apply(structure)
    r = {'site': site, 'damage': frac, 'seed': seed}
    for s, g in d.groupby('struct'):
        r[s] = round(100 * (g.m == 'True').mean(), 1)
    r['grammatical_%'] = round(100 * d.is_grammatical.mean(), 1)
    rows.append(r)
pd.DataFrame(rows)

done


,site,damage,seed,1 intransitive,2 transitive,3 ditransitive,copular,grammatical_%
0,grammar_A,0.4,1,78.7,86.3,75.0,82.6,94.0
1,grammar_A,0.5,1,76.0,80.4,44.2,78.3,85.2
2,grammar_A,0.4,2,83.1,82.2,21.2,82.6,92.2
3,grammar_A,0.5,2,82.0,90.4,48.1,100.0,94.4
4,control_meaning,0.4,0,29.0,48.9,50.0,80.4,91.2
5,control_meaning,0.5,0,26.2,47.5,50.0,76.1,90.8
